In [1]:
import pandas as pd
import os
import json
import numpy as np
from os.path import dirname

root_path = dirname(os.getcwd())

pd.set_option("display.max_columns", None)
data_dir = root_path + "/data/datasets/original/"
data_dir_processed = root_path + "/data/datasets/processed/"
data_dir_graphs = root_path + "/data/datasets/graphs_repair/"

print(root_path, data_dir, data_dir_processed, data_dir_graphs, sep="\n")

/home/matteo/Documents/GNN-test2/SEPH_MODELS/SEPH_TIME
/home/matteo/Documents/GNN-test2/SEPH_MODELS/SEPH_TIME/data/datasets/original/
/home/matteo/Documents/GNN-test2/SEPH_MODELS/SEPH_TIME/data/datasets/processed/
/home/matteo/Documents/GNN-test2/SEPH_MODELS/SEPH_TIME/data/datasets/graphs_repair/


In [2]:
with open("dataset_features.json", 'r') as file:
    datasets_info = json.load(file)


In [3]:
list(datasets_info.keys())

['BPI12_DECLINED_COMPLETE',
 'sepsis_cases_1',
 'sepsis_cases_4',
 'BPIC15_common']

In [4]:
dataset = "sepsis_cases_1"

In [5]:
tab_all = pd.read_csv(f"datasets/processed/{dataset}_processed_all.csv")
tab_all.head()

,Diagnose,DiagnosticArtAstrup,DiagnosticBlood,DiagnosticECG,DiagnosticIC,DiagnosticLacticAcid,DiagnosticLiquor,DiagnosticOther,DiagnosticSputum,DiagnosticUrinaryCulture,DiagnosticUrinarySediment,DiagnosticXthorax,DisfuncOrg,Hypotensie,Hypoxie,InfectionSuspected,Infusion,Oligurie,SIRSCritHeartRate,SIRSCritLeucos,SIRSCritTachypnea,SIRSCritTemperature,SIRSCriteria2OrMore,Age,CaseID,Activity,org:group,CRP,LacticAcid,Leucocytes,time:timestamp,timesincemidnight,month,weekday,hour,timesincelastevent,timesincecasestart,event_nr,open_cases,label,remaining_time
0,A,True,True,True,True,True,False,False,False,True,True,True,True,True,False,True,True,False,True,False,True,True,True,85.0,A,ER Registration,A,0.0,0.0,0.0,1.413955e+09,555,10,2,9,0.000000,0.000000,1,81,regular,968359.0
1,A,True,True,True,True,True,False,False,False,True,True,True,True,True,False,True,True,False,True,False,True,True,True,85.0,A,Leucocytes,B,0.0,0.0,9.6,1.413956e+09,567,10,2,9,0.000000,11.316667,2,81,regular,967680.0
2,A,True,True,True,True,True,False,False,False,True,True,True,True,True,False,True,True,False,True,False,True,True,True,85.0,A,CRP,B,21.0,0.0,9.6,1.413956e+09,567,10,2,9,0.000000,11.316667,3,81,regular,967680.0
3,A,True,True,True,True,True,False,False,False,True,True,True,True,True,False,True,True,False,True,False,True,True,True,85.0,A,LacticAcid,B,21.0,2.2,9.6,1.413956e+09,567,10,2,9,11.316667,11.316667,4,81,regular,967680.0
4,A,True,True,True,True,True,False,False,False,True,True,True,True,True,False,True,True,False,True,False,True,True,True,85.0,A,ER Triage,C,21.0,2.2,9.6,1.413956e+09,573,10,2,9,6.616667,17.933333,5,81,regular,967283.0


In [6]:
tab_train = pd.read_csv(f"datasets/processed/{dataset}_processed_train.csv")
tab_valid = pd.read_csv(f"datasets/processed/{dataset}_processed_valid.csv")
tab_test = pd.read_csv(f"datasets/processed/{dataset}_processed_test.csv")

In [7]:
if dataset.startswith("BPIC15"):
    with open("dataset_features.json", 'r') as file:
        dataset_info = json.load(file)["BPIC15_common"]
else:
    with open("dataset_features.json", 'r') as file:
        dataset_info = json.load(file)[dataset]

In [8]:
dataset_info

{'categorical': ['Diagnose',
  'DiagnosticArtAstrup',
  'DiagnosticBlood',
  'DiagnosticECG',
  'DiagnosticIC',
  'DiagnosticLacticAcid',
  'DiagnosticLiquor',
  'DiagnosticOther',
  'DiagnosticSputum',
  'DiagnosticUrinaryCulture',
  'DiagnosticUrinarySediment',
  'DiagnosticXthorax',
  'DisfuncOrg',
  'Hypotensie',
  'Hypoxie',
  'InfectionSuspected',
  'Infusion',
  'Oligurie',
  'SIRSCritHeartRate',
  'SIRSCritLeucos',
  'SIRSCritTachypnea',
  'SIRSCritTemperature',
  'SIRSCriteria2OrMore',
  'CaseID',
  'Activity',
  'org:group'],
 'numerical': ['Age',
  'CRP',
  'LacticAcid',
  'Leucocytes',
  'time:timestamp',
  'timesincemidnight',
  'month',
  'weekday',
  'hour',
  'timesincelastevent',
  'timesincecasestart',
  'event_nr',
  'open_cases']}

In [9]:
categorical_columns = dataset_info["categorical"]
real_value_columns = dataset_info["numerical"]

In [10]:
for k in categorical_columns:
    tab_all[k] = tab_all[k].astype("object")
    tab_train[k] = tab_train[k].astype("object")
    tab_valid[k] = tab_valid[k].astype("object")
    tab_test[k] = tab_test[k].astype("object")

### Prepare the graphs

In [11]:
import sklearn.preprocessing

from typing import List

In [12]:
def get_case_ids(tab):
    return list(tab["CaseID"].unique())

In [13]:
from torch import tensor, max, int64, float32
from torch_geometric.data import HeteroData

In [14]:
def get_one_hot_encoder(dataset: pd.DataFrame, key: str):
    datas = dataset[key].unique()
    datas = datas.reshape([len(datas), 1])
    onehot = sklearn.preprocessing.OneHotEncoder()
    onehot.fit(datas)
    return onehot

In [15]:
def get_one_hot_encodings(
    onehot, datas: pd.Series
):
    return onehot.transform(datas.reshape(-1, 1)).toarray()

In [16]:
def get_node_features(dataset: pd.DataFrame, trace: pd.DataFrame, cat_features, real_features) -> dict:
 

    res = {}

    for key in trace:
        values = trace[key].values
        if key in cat_features:
            onehot_encoder = get_one_hot_encoder(dataset, key)
            try:
                res[key] = tensor(
                    get_one_hot_encodings(onehot_encoder, values),
                    dtype=float32,
                    requires_grad=True
                )
            except ValueError:
                print(key)
                print(values)
        if key in real_features:
            res[key] = tensor(values,  dtype=float32,requires_grad=True)
            res[key] = res[key].reshape(res[key].shape[0], 1)
        
    

    return res


In [17]:


def compute_edges_indexs(node_features: dict, prefix_len):
    res = {}
    keys = node_features.keys()
    
    indexes = [[i, i + 1] for i in range(prefix_len-1)]
   
    for k in keys:
        if len(node_features[k]) != 1:
            if k == "Activity":
                res[(k, "followed_by", k)] = indexes
                for k2 in keys:
                    if k2 != k:
                        if len(node_features[k2]) == 1:
                            res[(k, "related_to", k2)] = [
                                [i, 0] for i in range(prefix_len)
                            ]
                        else:
                            res[(k, "related_to", k2)] = [
                                [i, i] for i in range(prefix_len)
                            ]
            else:
                res[(k, "related_to", k)] = indexes

    return res

In [18]:



def build_prefixes_graph_from_trace(dataset, trace, cat_features, real_features, prefix_length):
    X = []  # graphs
   
    
    
    node_features = get_node_features(dataset, trace, cat_features, real_features)
    
    
    
    
    G = HeteroData()
        
        
        
    for k in node_features:
        if k != "case:label":
            G[k].x = node_features[k][:prefix_length]


    edges_indexes = compute_edges_indexs(node_features, prefix_length)

    


    for k in edges_indexes:
        ce = [[], []]
        for i in range(len(edges_indexes[k])):
            ce[0].append(edges_indexes[k][i][0])
            ce[1].append(edges_indexes[k][i][1])
        edges_indexes[k] = ce

    for k in edges_indexes:
        G[k].edge_index = tensor(edges_indexes[k], dtype=int64)


    ## Get the label of the trace
    label_value = trace["remaining_time"].values[prefix_length -1]
    G.y = tensor([label_value], dtype=float32)
    
        
    X.append(G)
    
    return X

## Create the datasets

In [19]:
case_train_ids = get_case_ids(tab_train)
case_valid_ids = get_case_ids(tab_valid)
case_test_ids = get_case_ids(tab_test)

In [20]:
print(len(case_train_ids))
print(len(case_valid_ids))
print(len(case_test_ids))

416
105
261


In [21]:
tab_train["CaseID"] = tab_train["CaseID"].astype(np.str_)
tab_valid["CaseID"] = tab_valid["CaseID"].astype(np.str_)
tab_test["CaseID"] = tab_test["CaseID"].astype(np.str_)

In [22]:
trace = (
        tab_train.query(f"CaseID == '{case_train_ids[0]}'")
        .reset_index()
        .drop(columns="index")
        .drop(columns="CaseID")
    )
trace 

,Diagnose,DiagnosticArtAstrup,DiagnosticBlood,DiagnosticECG,DiagnosticIC,DiagnosticLacticAcid,DiagnosticLiquor,DiagnosticOther,DiagnosticSputum,DiagnosticUrinaryCulture,DiagnosticUrinarySediment,DiagnosticXthorax,DisfuncOrg,Hypotensie,Hypoxie,InfectionSuspected,Infusion,Oligurie,SIRSCritHeartRate,SIRSCritLeucos,SIRSCritTachypnea,SIRSCritTemperature,SIRSCriteria2OrMore,Age,Activity,org:group,CRP,LacticAcid,Leucocytes,time:timestamp,timesincemidnight,month,weekday,hour,timesincelastevent,timesincecasestart,event_nr,open_cases,label,remaining_time
0,B,False,False,True,True,True,False,False,False,False,True,True,False,False,False,True,True,False,False,False,True,True,True,70.0,ER Registration,A,0.0,0.0,0.0,1.387426e+09,484,12,3,8,0.000000,0.000000,1,22,regular,1392022.0
1,B,False,False,True,True,True,False,False,False,False,True,True,False,False,False,True,True,False,False,False,True,True,True,70.0,ER Triage,C,0.0,0.0,0.0,1.387426e+09,493,12,3,8,8.883333,8.883333,2,22,regular,1391489.0
2,B,False,False,True,True,True,False,False,False,False,True,True,False,False,False,True,True,False,False,False,True,True,True,70.0,ER Sepsis Triage,A,0.0,0.0,0.0,1.387426e+09,493,12,3,8,0.383333,9.266667,3,22,regular,1391466.0
3,B,False,False,True,True,True,False,False,False,False,True,True,False,False,False,True,True,False,False,False,True,True,True,70.0,LacticAcid,B,0.0,1.1,0.0,1.387427e+09,506,12,3,8,0.000000,21.366667,4,22,regular,1390740.0
4,B,False,False,True,True,True,False,False,False,False,True,True,False,False,False,True,True,False,False,False,True,True,True,70.0,CRP,B,102.0,1.1,0.0,1.387427e+09,506,12,3,8,0.000000,21.366667,5,22,regular,1390740.0
5,B,False,False,True,True,True,False,False,False,False,True,True,False,False,False,True,True,False,False,False,True,True,True,70.0,Leucocytes,B,102.0,1.1,11.3,1.387427e+09,506,12,3,8,12.100000,21.366667,6,22,regular,1390740.0
6,B,False,False,True,True,True,False,False,False,False,True,True,False,False,False,True,True,False,False,False,True,True,True,70.0,IV Liquid,A,102.0,1.1,11.3,1.387430e+09,558,12,3,9,52.733333,74.100000,7,22,regular,1387576.0
7,B,False,False,True,True,True,False,False,False,False,True,True,False,False,False,True,True,False,False,False,True,True,True,70.0,IV Antibiotics,A,102.0,1.1,11.3,1.387432e+09,585,12,3,9,27.150000,101.250000,8,22,regular,1385947.0
8,B,False,False,True,True,True,False,False,False,False,True,True,False,False,False,True,True,False,False,False,True,True,True,70.0,Admission NC,S,102.0,1.1,11.3,1.387442e+09,754,12,3,12,168.800000,270.050000,9,22,regular,1375819.0
9,B,False,False,True,True,True,False,False,False,False,True,True,False,False,False,True,True,False,False,False,True,True,True,70.0,Admission IC,P,102.0,1.1,11.3,1.387469e+09,1205,12,3,20,450.583333,720.633333,10,23,regular,1348784.0


In [23]:
min_len = tab_all.groupby("CaseID").size().min()
max_len = tab_all.groupby("CaseID").size().max()
print("Minimum trace length:", min_len)
print("Maximum trace length:", max_len)

Minimum trace length: 5
Maximum trace length: 185


In [24]:
import pickle
from tqdm.notebook import tqdm

In [25]:
PREFIX_LENGTH = 4

In [26]:
print("Preparing training dataset...")

X_train = []


for i in tqdm(range(len(case_train_ids))):
    trace = (
        tab_train.query(f"CaseID == '{case_train_ids[i]}'")
        .reset_index(drop=True)
        .drop(columns="CaseID")
    )

    if len(trace) >= PREFIX_LENGTH:
        graphs = build_prefixes_graph_from_trace(
            dataset=tab_all,
            trace=trace,
            cat_features=categorical_columns,
            real_features=real_value_columns,
            prefix_length=PREFIX_LENGTH,
        )
        for j in range(len(graphs)):
            X_train.append(graphs[j])

Preparing training dataset...


  0%|          | 0/416 [00:00<?, ?it/s]

In [27]:
with open(data_dir_graphs + dataset + "_TRAIN_repair.pkl", "wb") as f:
    pickle.dump(X_train, f)

In [28]:
print("Preparing validation dataset...")

X_valid = []


for i in tqdm(range(len(case_valid_ids))):
    trace = (
        tab_valid.query(f"CaseID == '{case_valid_ids[i]}'")
        .reset_index(drop=True)
        .drop(columns="CaseID")
    )
    if len(trace) >= PREFIX_LENGTH:
        graphs = build_prefixes_graph_from_trace(
            dataset=tab_all,
            trace=trace,
            cat_features=categorical_columns,
            real_features=real_value_columns,
            prefix_length=PREFIX_LENGTH
        )
        for j in range(len(graphs)):
            X_valid.append(graphs[j])

Preparing validation dataset...


  0%|          | 0/105 [00:00<?, ?it/s]

In [29]:
with open(data_dir_graphs + dataset + "_VALID_repair.pkl", "wb") as f:
    pickle.dump(X_valid, f)

In [30]:
print("Preparing test dataset...")

X_test = []


for i in tqdm(range(len(case_test_ids))):
    trace = (
        tab_test.query(f"CaseID == '{case_test_ids[i]}'")
        .reset_index(drop=True)
        .drop(columns="CaseID")
    )

    if len(trace) >= PREFIX_LENGTH:
        graphs = build_prefixes_graph_from_trace(
            dataset=tab_all,
            trace=trace,
            cat_features=categorical_columns,
            real_features=real_value_columns,
            prefix_length=PREFIX_LENGTH
        )
        for j in range(len(graphs)):
            X_test.append(graphs[j])

Preparing test dataset...


  0%|          | 0/261 [00:00<?, ?it/s]

In [31]:
with open(data_dir_graphs + dataset + "_TEST_repair.pkl", "wb") as f:
    pickle.dump(X_test, f)